In [1]:
import re, string, json, os, numpy as np
from collections import Counter
from rouge import Rouge

def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)
    def white_space_fix(text):
        return " ".join(text.split())
    def remove_punc(text):
        return "".join(ch for ch in text if ch not in set(string.punctuation))
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def f1_score(prediction, ground_truth):
    common = Counter(prediction) & Counter(ground_truth)
    num_same = sum(common.values())
    if num_same == 0: return 0
    precision = 1.0 * num_same / len(prediction)
    recall = 1.0 * num_same / len(ground_truth)
    return (2 * precision * recall) / (precision + recall)

def qa_f1_score(prediction, ground_truth, **kwargs):
    return f1_score(normalize_answer(prediction).split(), normalize_answer(ground_truth).split())

def rouge_score(prediction, ground_truth, **kwargs):
    try:
        return Rouge().get_scores([prediction], [ground_truth], avg=True)["rouge-l"]["f"]
    except:
        return 0.0

dataset2metric = {
    "narrativeqa": qa_f1_score, "qasper": qa_f1_score,
    "multifieldqa_en": qa_f1_score, "hotpotqa": qa_f1_score,
    "2wikimqa": qa_f1_score, "musique": qa_f1_score,
    "gov_report": rouge_score, "qmsum": rouge_score,
    "multi_news": rouge_score, "samsum": rouge_score,
}

def score_dataset(jsonl_path):
    with open(jsonl_path) as f:
        data = [json.loads(line) for line in f]
    dataset = os.path.splitext(os.path.basename(jsonl_path))[0]
    metric = dataset2metric.get(dataset, qa_f1_score)
    scores = []
    for item in data:
        best = 0
        for gt in item["answers"]:
            best = max(best, metric(item["pred"], gt))
        scores.append(best)
    return dataset, round(100 * np.mean(scores), 2)



In [2]:
import os

vanilla_dir = "pred/vanilla_run/"
adakv_dir = "pred/adakv_run/"

results = []
for fname in os.listdir(vanilla_dir):
    if not fname.endswith(".jsonl"):
        continue
    
    ds = fname.replace(".jsonl", "")
    van_path = os.path.join(vanilla_dir, fname)
    ada_path = os.path.join(adakv_dir, fname)
    
    _, van_score = score_dataset(van_path)
    
    if os.path.exists(ada_path):
        _, ada_score = score_dataset(ada_path)
        delta = ada_score - van_score
    else:
        ada_score = None
        delta = None
    
    results.append((ds, van_score, ada_score, delta))

# Print comparison table
print(f"{'Dataset':<20} {'Vanilla':>8} {'AdaKV':>8} {'Δ':>8}")
print("-" * 48)
van_scores = []
for ds, v, a, d in sorted(results, key=lambda x: x[3] if x[3] is not None else 0):
    van_scores.append(v)
    a_str = f"{a:>8.2f}" if a is not None else "   N/A  "
    d_str = f"{d:>+8.2f}" if d is not None else "   N/A  "
    print(f"{ds:<20} {v:>8.2f} {a_str} {d_str}")

print("-" * 48)
avg_v = np.mean(van_scores)
completed = [r for r in results if r[2] is not None]
avg_a = np.mean([r[2] for r in completed]) if completed else 0
print(f"{'AVERAGE':<20} {avg_v:>8.2f} {'N/A':>8}" if not completed else
      f"{'AVERAGE':<20} {avg_v:>8.2f} {avg_a:>8.2f} {avg_a-avg_v:>+8.2f}")

Dataset               Vanilla    AdaKV        Δ
------------------------------------------------
gov_report              28.09    14.43   -13.66
multi_news               4.94     2.33    -2.61
qasper                  12.51    11.53    -0.98
narrativeqa             18.77    18.13    -0.64
hotpotqa                 8.35     8.17    -0.18
samsum                  30.08    34.22    +4.14
------------------------------------------------
AVERAGE                 17.12    14.80    -2.32


In [2]:
import os, numpy as np

vanilla_dir = "pred/vanilla_run/"
adakv_dir = "pred/adakv_run/"
entropy_dir = "pred/entropy_run/"

results = []
for fname in os.listdir(vanilla_dir):
    if not fname.endswith(".jsonl"):
        continue
    
    ds = fname.replace(".jsonl", "")
    van_path = os.path.join(vanilla_dir, fname)
    ada_path = os.path.join(adakv_dir, fname)
    ent_path = os.path.join(entropy_dir, fname)
    
    _, van_score = score_dataset(van_path)
    _, ada_score = score_dataset(ada_path) if os.path.exists(ada_path) else (None, None)
    _, ent_score = score_dataset(ent_path) if os.path.exists(ent_path) else (None, None)
    
    results.append((ds, van_score, ada_score, ent_score))

# Print comparison table
print(f"{'Dataset':<20} {'Vanilla':>8} {'AdaKV':>8} {'H2O(ent)':>8} {'Δada':>8} {'Δent':>8}")
print("-" * 68)
van_scores = []
for ds, v, a, e in sorted(results, key=lambda x: (x[2] or 0) - x[1]):
    van_scores.append(v)
    a_str = f"{a:>8.2f}" if a is not None else "   N/A  "
    e_str = f"{e:>8.2f}" if e is not None else "   N/A  "
    d_ada = f"{a-v:>+8.2f}" if a is not None else "   N/A  "
    d_ent = f"{e-v:>+8.2f}" if e is not None else "   N/A  "
    print(f"{ds:<20} {v:>8.2f} {a_str} {e_str} {d_ada} {d_ent}")

print("-" * 68)
avg_v = np.mean(van_scores)
completed_ada = [r for r in results if r[2] is not None]
completed_ent = [r for r in results if r[3] is not None]
avg_a = np.mean([r[2] for r in completed_ada]) if completed_ada else 0
avg_e = np.mean([r[3] for r in completed_ent]) if completed_ent else 0
print(f"{'AVERAGE':<20} {avg_v:>8.2f} {avg_a:>8.2f} {avg_e:>8.2f} {avg_a-avg_v:>+8.2f} {avg_e-avg_v:>+8.2f}")

Dataset               Vanilla    AdaKV H2O(ent)     Δada     Δent
--------------------------------------------------------------------
gov_report              28.09    13.98    N/A     -14.11    N/A  
multi_news               4.94     1.48    N/A      -3.46    N/A  
qasper                  12.51    11.17    11.17    -1.34    -1.34
qasper_preds            12.51    11.17    N/A      -1.34    N/A  
narrativeqa             18.77    17.77    17.77    -1.00    -1.00
hotpotqa                 8.35     8.45     8.45    +0.10    +0.10
samsum                  30.08    38.09    N/A      +8.01    N/A  
--------------------------------------------------------------------
AVERAGE                 16.46    14.59    12.46    -1.88    -4.00


In [2]:
import os, numpy as np

BUDGET = 128  # ← change to the budget you just ran

vanilla_dir = "pred/vanilla_run/"
adakv_dir = f"pred/sweep/b{BUDGET}/adakv/"
entropy_dir = f"pred/sweep/b{BUDGET}/entropy/"

results = []
for fname in os.listdir(vanilla_dir):
    if not fname.endswith(".jsonl"):
        continue
    
    ds = fname.replace(".jsonl", "")
    van_path = os.path.join(vanilla_dir, fname)
    ada_path = os.path.join(adakv_dir, fname)
    ent_path = os.path.join(entropy_dir, fname)
    
    _, van_score = score_dataset(van_path)
    _, ada_score = score_dataset(ada_path) if os.path.exists(ada_path) else (None, None)
    _, ent_score = score_dataset(ent_path) if os.path.exists(ent_path) else (None, None)
    
    results.append((ds, van_score, ada_score, ent_score))

print(f"{'Dataset':<20} {'Vanilla':>8} {'AdaKV':>8} {'H2O(ent)':>8} {'Δada':>8} {'Δent':>8}")
print(f"{'─'*68}")
van_scores = []
for ds, v, a, e in sorted(results, key=lambda x: (x[2] or 0) - x[1]):
    van_scores.append(v)
    a_str = f"{a:>8.2f}" if a is not None else "   N/A  "
    e_str = f"{e:>8.2f}" if e is not None else "   N/A  "
    d_ada = f"{a-v:>+8.2f}" if a is not None else "   N/A  "
    d_ent = f"{e-v:>+8.2f}" if e is not None else "   N/A  "
    print(f"{ds:<20} {v:>8.2f} {a_str} {e_str} {d_ada} {d_ent}")

print(f"{'─'*68}")
avg_v = np.mean(van_scores)
completed_ada = [r for r in results if r[2] is not None]
completed_ent = [r for r in results if r[3] is not None]
avg_a = np.mean([r[2] for r in completed_ada]) if completed_ada else 0
avg_e = np.mean([r[3] for r in completed_ent]) if completed_ent else 0
print(f"{'AVERAGE':<20} {avg_v:>8.2f} {avg_a:>8.2f} {avg_e:>8.2f} {avg_a-avg_v:>+8.2f} {avg_e-avg_v:>+8.2f}")
print(f"\nBudget = {BUDGET} | Vanilla reused from {vanilla_dir}")

Dataset               Vanilla    AdaKV H2O(ent)     Δada     Δent
────────────────────────────────────────────────────────────────────
gov_report              28.09    12.13    12.13   -15.96   -15.96
qasper_preds            12.51    N/A      N/A      N/A      N/A  
qasper                  12.51     6.92     6.92    -5.59    -5.59
multi_news               4.94     1.20     1.20    -3.74    -3.74
narrativeqa             18.77    15.32    13.10    -3.45    -5.67
hotpotqa                 8.35     8.47     8.47    +0.12    +0.12
samsum                  30.08    39.86    39.86    +9.78    +9.78
────────────────────────────────────────────────────────────────────
AVERAGE                 16.46    13.98    13.61    -2.48    -2.85

Budget = 128 | Vanilla reused from pred/vanilla_run/


In [4]:
import os, numpy as np

BUDGET = 128  # ← change to the budget you just ran

vanilla_dir = "pred/vanilla_run/"
adakv_dir = f"pred/sweep/b{BUDGET}/adakv/"
entropy_dir = f"pred/sweep/b{BUDGET}/entropy/"
entropy_full_chunked = f"pred/sweep/b{BUDGET}/entropy_fullchunk/"

results = []
for fname in os.listdir(vanilla_dir):
    if not fname.endswith(".jsonl"):
        continue
    
    ds = fname.replace(".jsonl", "")
    van_path = os.path.join(vanilla_dir, fname)
    ada_path = os.path.join(adakv_dir, fname)
    ent_path = os.path.join(entropy_dir, fname)
    ent_full_path = os.path.join(entropy_full_chunked, fname)
    
    _, van_score = score_dataset(van_path)
    _, ada_score = score_dataset(ada_path) if os.path.exists(ada_path) else (None, None)
    _, ent_score = score_dataset(ent_path) if os.path.exists(ent_path) else (None, None)

    _, ent_full_score = score_dataset(ent_full_path) if os.path.exists(ent_full_path) else (None, None)

    results.append((ds, van_score, ada_score, ent_score, ent_full_score))


print(f"{'Dataset':<20} {'Vanilla':>8} {'AdaKV':>8} {'H2O(ent)':>8} {'H2O(ent_full)':>12} {'Δada':>8} {'Δent':>8} {'Δent_full':>12}")
print(f"{'─'*92}")
van_scores = []
for ds, v, a, e, ef in sorted(results, key=lambda x: (x[2] or 0) - x[1]):
    van_scores.append(v)
    a_str = f"{a:>8.2f}" if a is not None else "   N/A  "
    e_str = f"{e:>8.2f}" if e is not None else "   N/A  "
    ef_str = f"{ef:>12.2f}" if ef is not None else "      N/A      "
    d_ada = f"{a-v:>+8.2f}" if a is not None else "   N/A  "
    d_ent = f"{e-v:>+8.2f}" if e is not None else "   N/A  "
    d_ent_full = f"{ef-v:>+12.2f}" if ef is not None else "      N/A      "
    print(f"{ds:<20} {v:>8.2f} {a_str} {e_str} {ef_str} {d_ada} {d_ent} {d_ent_full}")

print(f"{'─'*92}")
avg_v = np.mean(van_scores)
completed_ada = [r for r in results if r[2] is not None]
completed_ent = [r for r in results if r[3] is not None]
completed_ent_full = [r for r in results if r[4] is not None]
avg_a = np.mean([r[2] for r in completed_ada]) if completed_ada else 0
avg_e = np.mean([r[3] for r in completed_ent]) if completed_ent else 0
avg_ef = np.mean([r[4] for r in completed_ent_full]) if completed_ent_full else 0
print(f"{'AVERAGE':<20} {avg_v:>8.2f} {avg_a:>8.2f} {avg_e:>8.2f} {avg_a-avg_v:>+8.2f} {avg_e-avg_v:>+8.2f}")
print(f"\nBudget = {BUDGET} | Vanilla reused from {vanilla_dir}")

Dataset               Vanilla    AdaKV H2O(ent) H2O(ent_full)     Δada     Δent    Δent_full
────────────────────────────────────────────────────────────────────────────────────────────
gov_report              28.09    12.13    N/A          11.33   -15.96    N/A         -16.76
qasper                  12.51     6.92    N/A           6.58    -5.59    N/A          -5.93
multi_news               4.94     1.20    N/A           1.70    -3.74    N/A          -3.24
narrativeqa             18.77    15.32    N/A          12.66    -3.45    N/A          -6.11
hotpotqa                 8.35     8.47    N/A           8.06    +0.12    N/A          -0.29
samsum                  30.08    39.86    N/A          39.14    +9.78    N/A          +9.06
────────────────────────────────────────────────────────────────────────────────────────────
AVERAGE                 17.12    13.98     0.00    -3.14   -17.12

Budget = 128 | Vanilla reused from pred/vanilla_run/
